In [1]:
import os
import re
import json
import pandas as pd
from io import StringIO
from dotenv import load_dotenv
load_dotenv()
from scholarlm.utils import get_filenames_in_directory, table_extract, process_pdf

%load_ext autoreload
%autoreload 2

INFO 12-17 21:10:34 [__init__.py:220] No platform detected, vLLM is running on UnspecifiedPlatform
WARNING 12-17 21:10:39 [_custom_ops.py:20] Failed to import from vllm._C with ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


In [2]:
main_directory = os.getenv("POND_PATH")
pdf_directory = os.getenv("POND_PDF_PATH")
text_directory = os.getenv("POND_TEXT_PATH")

with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)

text_files = get_filenames_in_directory(text_directory, ignore = [".DS_Store"])
text_files.sort()

text_files = [
    'physical_and_chemical_limnological.txt',
    'physical-chemical_influences.txt',
    'prairie_wetland.txt',
    'net_heterotrophy.txt',
    'habitat_characteristics.txt',
    'biodiversity_of_constructed.txt',
    'fish_production_in_lakes.txt',
    'long-term_stability.txt',
    'diversity_of_macroinvertebrates.txt',
    'impact_of_macrophytes.txt'
]


text_filepaths = []
text_info = []
for f in text_files:
    paper_code = f.replace('.txt', '')
    filepath = os.path.join(text_directory, f)
    metadata = paper_info.get(paper_code, {})
    # ID Addition:
    metadata['paper_code'] = paper_code
    text_filepaths.append(filepath)
    text_info.append(metadata)

In [5]:
with open(text_filepaths[2], "r") as f:
    sample_text = f.read()

tables = pd.read_html(StringIO(sample_text))

In [6]:
print(sample_text)

<page number="0">

Prairie wetland communities recover at different rates following hydrological restoration

LAUREN E. BORTOLOTTI, ROLF D. VINEBROOKE AND VINCENT L. ST. LOUIS
Department of Biological Sciences, University of Alberta, Edmonton, AB, Canada

SUMMARY

1. Prairie pothole wetlands provide many ecosystem services, including supporting biodiversity and filtering water on the landscape. However, over half of these wetlands have been drained for agriculture, thereby requiring restoration to re-establish ecosystem services.
2. We assessed the recovery of hydrologically restored wetlands based on water chemistry and taxonomic shifts within and across five biological communities (phytoplankton, benthic diatoms, zooplankton, macroinvertebrates, submersed aquatic vegetation [SAV]). We sampled 24 wetlands in southeastern Saskatchewan, Canada, spanning three restoration states: recently restored (restored 1–3 years before the study; \( n = 8 \)), older restored (restored 7–14 years bef

In [10]:
tables[1].iloc[:, :20]

,Site,Restoration state,SpCond (μS cm−1),pH,NH4+ (μg L−1),NO2+NO3− (μg L−1),TDN (μg L−1),TP (μg L−1),TDP (μg L−1),DOC (mg L−1),CO2 (μmol L−1),DIC (μmol L−1),Chl a (μg L−1),TSS (mg L−1),Unnamed: 14_level_0,Unnamed: 15_level_0,Unnamed: 16_level_0,Unnamed: 17_level_0,Unnamed: 18_level_0,Unnamed: 19_level_0
,Site,Restoration state,SpCond (μS cm−1),pH,NH4+ (μg L−1),NO2+NO3− (μg L−1),TDN (μg L−1),TP (μg L−1),TDP (μg L−1),DOC (mg L−1),CO2 (μmol L−1),DIC (μmol L−1),Chl a (μg L−1),TSS (mg L−1),Unnamed: 14_level_1,Unnamed: 15_level_1,Unnamed: 16_level_1,Unnamed: 17_level_1,Unnamed: 18_level_1,Unnamed: 19_level_1
0,Hines,RR,"438 (338, 502)","7.44 (7.09, 7.86)","78 (8, 137)","2 (2, 2)","1550 (810, 2020)","637 (418, 765)","562 (365, 673)","15.1 (3.2, 28.5)","307 (304, 309)","3170 (3099, 3241)","12.5 (4.2, 27.1)","4.9 (2.0, 7.5)",NaN,NaN,NaN,NaN,NaN,NaN
1,Hood-1,RR,"580 (543, 616)","7.57 (7.18, 7.96)","8 (7, 9)","1 (0, 1)","1099 (792, 1346)","44 (29, 67)","36 (29, 40)","26.8 (16.1, 40.2)","418 (361, 475)","3485 (3438, 3532)","9.3 (1.3, 15.8)","1.6 (0.0, 3.5)",NaN,NaN,NaN,NaN,NaN,NaN
2,Hood-2,RR,"537 (500, 574)","7.83 (7.42, 8.24)","27 (7, 63)","1 (0, 2)","999 (760, 1120)","50 (29, 75)","34 (21, 42)","16.1 (13.4, 20.7)","187 (168, 205)","3332 (3253, 3412)","8.2 (1.0, 20.8)","1.5 (0.0, 4.0)",NaN,NaN,NaN,NaN,NaN,NaN
3,Johanson,RR,"1114 (946, 1283)","7.38 (7.22, 7.54)","25 (16, 34)","1 (0, 2)","1971 (1542, 2400)","293 (140, 445)","185 (128, 242)","32.6 (25.8, 39.3)","427 (25.8, 39.3)","4185 (2.3, 124.1)","63.2 (1.6, 7.6)","4.6 (1.6, 7.6)",NaN,NaN,NaN,NaN,NaN,NaN
4,Reinson,RR,"1909 (1690, 2046)","7.52 (7.37, 7.67)","120 (14, 197)","1 (0, 3)","2415 (1646, 2920)","608 (487, 700)","523 (429, 622)","35.3 (27.4, 43.2)","455 (365, 545)","4149 (4111, 4186)","74.5 (13.7, 129.0)","16.7 (1.2, 41.0)",NaN,NaN,NaN,NaN,NaN,NaN
5,Smith-1,RR,"308 (291, 327)","7.09 (6.67, 7.79)","11 (6, 18)","1 (1, 2)","769 (738, 812)","78 (41, 141)","49 (33, 61)","11.8 (9.3, 14.2)","367 (319, 414)","2618 (2561, 2676)","17.2 (4.2, 38.4)","4.3 (0.8, 8.4)",NaN,NaN,NaN,NaN,NaN,NaN
6,Smith-2,RR,"418 (386, 461)","6.83 (6.67, 7.05)","95 (16, 150)","2 (1, 3)","1133 (1068, 1248)","199 (148, 238)","168 (74, 227)","18.8 (17.3, 19.8)","601 (584, 618)","3066 (3056, 3076)","4.9 (4.0, 7.0)","6.8 (0.8, 14.0)",NaN,NaN,NaN,NaN,NaN,NaN
7,Sorrell,RR,"1120 (963, 1200)","7.57 (7.16, 8.01)","62 (9, 149)","1 (0, 1)","1767 (1214, 2120)","78 (57, 91)","61 (41, 73)","22.7 (15.3, 33.7)","260 (256, 265)","3514 (3460, 3569)","7.1 (3.2, 12.5)","1.2 (0.4, 2.5)",NaN,NaN,NaN,NaN,NaN,NaN
8,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,Restoration state mean,803.0,7.4,53.0,1.0,1463.0


In [14]:
with open("../data/12_17_25/habitat_standardize.json", "r") as f:
    result_dict = json.load(f)

In [15]:
len(result_dict)

1

In [16]:
result_dict

[{'title': 'habitat characteristics and odonate diversity in mountain ponds of central italy',
  'author': 'carchini et al.',
  'year': 2005,
  'paper_code': 'habitat_characteristics',
  'document_id': 0,
  'context': '<page number="0">\n\nSee discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/229087071\n\nHabitat characteristics and odonate diversity in mountain ponds of central Italy\n\nArticle in Aquatic Conservation Marine and Freshwater Ecosystems · November 2005\nDOI: 10.1002/aqc.741\n\nCITATIONS\n22\n\nREADS\n97\n\n3 authors, including:\n\nG. Carchini\nUniversity of Rome Tor Vergata\n51 PUBLICATIONS  514 CITATIONS\nSEE PROFILE\n\nAngelo G Solimini\nSapienza University of Rome\n95 PUBLICATIONS  1,665 CITATIONS\nSEE PROFILE\n\n</page>\n\n<page number="1">\n\n***IMMEDIATE RESPONSE REQUIRED***\n\nYour article may be published online via Wiley\'s EarlyView® service (www.interscience.wiley.com) shortly after receipt of corrections

In [67]:
bc = [r for r in result_dict if r['name'] == 'BC']

In [68]:
bc

[{'document_id': 0,
  'context': '<page number="0">\n\n1Darlene S. S. Lim, 1Marianne S. V. Douglas, 2John P. Smol and 3David R. S. Lean\n\n1Paleoenvironmental Assessment Laboratory (PAL), Department of Geology, 22 Russell St., University of Toronto, Toronto, Ontario, M5S 3B1, Canada\n2Paleoecological Environmental Assessment and Research Laboratory (PEARL), Department of Biology, Queen’s University, Kingston, Ontario, K7L 3N6, Canada\n3Department of Biology, University of Ottawa, P.O. Box 450 Station A, Ottawa, Ontario, K1N 6N5, Canada\n\nPhysical and Chemical Limnological Characteristics of 38 Lakes and Ponds on Bathurst Island, Nunavut, Canadian High Arctic\n\nkey words: limnology, high arctic, nitrogen, phosphorus, dissolved organic carbon\n\nAbstract\n\nThe limnological features that characterize the shallow ponds (<2 m deep) and lakes (>2 m deep) on Bathurst Island, Nunavut, Canada were examined through chemical analyses and multivariate statistical methods as part of a larger on-

In [198]:
rs = [r for r in result_dict if r.get('value') is not None]

In [199]:
rs

[]

In [192]:
len(rs)

16

In [130]:
from enum import Enum
from pydantic import BaseModel

def create_multichoice_schema(choices: list[str]):
    # Dynamically create an Enum from your choices
    ChoiceEnum = Enum('ChoiceEnum', {choice: choice for choice in choices}, type=str)
    
    class MultiChoice(BaseModel):
        selections: list[ChoiceEnum]
    
    return MultiChoice

In [133]:
create_multichoice_schema(cell_contents).model_fields.keys()

dict_keys(['selections'])